# Phase 5: MLOps Best Practices
**Goals:**
- Automate data processing using a single end-to-end Spark ML Pipeline
- Track model experiments (hyperparameters, metrics, artefacts) with **MLflow**
- Compare Logistic Regression vs GBT runs and register the best model

In [4]:
# Local Spark session (run this first when running outside Databricks)
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Phase5_MLOps") \
    .master("local[*]") \
    .getOrCreate()
print("Spark session ready:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/04 21:14:17 WARN Utils: Your hostname, Shubhs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 100.70.79.150 instead (on interface en0)
26/03/04 21:14:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 21:14:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/04 21:14:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark session ready: 4.1.1


In [5]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from functools import reduce
import re, mlflow, mlflow.spark
from mlflow.models.signature import infer_signature

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Databricks-native MLflow tracking 
mlflow.set_experiment('/Shared/phase5_campaign_model')
print('MLflow tracking URI:', mlflow.get_tracking_uri())

MLflow tracking URI: sqlite:////Users/shubhvaishnav/Desktop/590/590/mlflow.db


### 1) Automated Data Ingestion

In [6]:
paths = {
    "nykaa":   "/Users/shubhvaishnav/Desktop/590/590/data/nykaa_campaign_data.csv",
    "purplle": "/Users/shubhvaishnav/Desktop/590/590/data/purplle_campaign_data.csv",
    "tira":    "/Users/shubhvaishnav/Desktop/590/590/data/tira_campaign_data.csv"
}

def normalize_colname(c: str) -> str:
    c = c.strip().lower()
    c = re.sub(r'[^a-z0-9]+', '_', c)
    c = re.sub(r'_+', '_', c).strip('_')
    return c

def read_campaign(path: str, brand: str):
    df = (spark.read
          .option('header', True)
          .option('inferSchema', True)
          .csv(path))
    for old in df.columns:
        df = df.withColumnRenamed(old, normalize_colname(old))
    df = df.withColumn('brand_source', F.lit(brand))
    return df

def align_cols(df, all_cols):
    for c in all_cols:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None))
    return df.select(*all_cols)

def load_all_brands(paths: dict):
    dfs = [read_campaign(p, b) for b, p in paths.items()]
    all_cols = sorted(set().union(*[set(d.columns) for d in dfs]))
    dfs_aligned = [align_cols(d, all_cols) for d in dfs]
    df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_aligned)
    return df.dropDuplicates()

df_raw = load_all_brands(paths)
print(f'Loaded {df_raw.count():,} rows across {len(df_raw.columns)} columns')

Loaded 166,665 rows across 17 columns


### 2) Feature Engineering (reusable function)

In [7]:
def engineer_features(df):
    """All feature-engineering steps from Phase 4, now in one callable function."""

    # --- Channel flags ---
    df = df.withColumn('channel_used', F.col('channel_used').cast('string'))
    df = df.withColumn('channels_array', F.split(F.col('channel_used'), ',\\s*'))

    channels = (
        df.select(F.explode('channels_array').alias('ch'))
          .where(F.col('ch').isNotNull() & (F.trim(F.col('ch')) != ''))
          .select(F.trim(F.col('ch')).alias('ch'))
          .distinct().toPandas()['ch'].tolist()
    )
    for ch in channels:
        safe = normalize_colname(ch)
        df = df.withColumn(
            f'ch_{safe}',
            F.when(F.array_contains(F.col('channels_array'), ch), 1).otherwise(0)
        )
    df = df.drop('channels_array')

    # --- ROAS ---
    df = df.withColumn('roas', F.col('revenue') / F.col('acquisition_cost'))

    # --- Date parts (data is DD-MM-YYYY e.g. '06-07-2024') ---
    df = df.withColumn('date', F.to_date(F.col('date'), 'dd-MM-yyyy'))
    df = (df
          .withColumn('year',      F.year('date'))
          .withColumn('month',     F.month('date'))
          .withColumn('dayofweek', F.dayofweek('date')))

    # --- Binary label: high-performing campaign (ROAS > median) ---
    # Use approxQuantile for efficiency on large data
    median_roas = df.approxQuantile('roas', [0.5], 0.01)[0]
    df = df.withColumn('label', (F.col('roas') > median_roas).cast('double'))

    return df, channels

df_feat, channels = engineer_features(df_raw)
print('Feature engineering complete. Label distribution:')
display(df_feat.groupBy('label').count())

Feature engineering complete. Label distribution:


DataFrame[label: double, count: bigint]

### 3) Spark ML Pipeline Definition

In [8]:
categorical_cols = ['brand_source', 'campaign_type', 'customer_segment', 'language']

channel_cols = [c for c in df_feat.columns if c.startswith('ch_')]
numeric_cols = [
    'acquisition_cost', 'revenue', 'conversion_rate',
    'clicks', 'impressions', 'year', 'month', 'dayofweek'
] + channel_cols

def build_pipeline(classifier):
    """Return a full Pipeline: encoding → assembly → model."""
    indexers  = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep')
                 for c in categorical_cols]
    encoders  = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_vec')
                 for c in categorical_cols]
    cat_vec_cols = [c+'_vec' for c in categorical_cols]

    assembler = VectorAssembler(
        inputCols=numeric_cols + cat_vec_cols,
        outputCol='features',
        handleInvalid='skip'
    )
    stages = indexers + encoders + [assembler, classifier]
    return Pipeline(stages=stages)

print('Pipeline builder ready. Numeric features:', len(numeric_cols),
      '| Categorical cols:', categorical_cols)

Pipeline builder ready. Numeric features: 14 | Categorical cols: ['brand_source', 'campaign_type', 'customer_segment', 'language']


In [9]:
train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,}  |  Test: {test_df.count():,}')

26/03/04 21:14:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/04 21:14:29 ERROR Executor: Exception in task 8.0 in stage 26.0 (TID 74)6]
org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '06-07-2024' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"year" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)

	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeErrorInternal(ExecutionErrors.scala:115)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeErrorInternal$(ExecutionErrors.scala:102)
	at org.apache.spark.sql.errors.ExecutionErrors$.invalidInputInCastToDat

DateTimeException: [CAST_INVALID_INPUT] The value '24-08-2024' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"year" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)


### 4)  MLflow Experiment Tracking

4a  Logistic Regression

In [0]:
evaluator = BinaryClassificationEvaluator(metricName='areaUnderROC')

lr_params = [
    {'regParam': 0.01, 'elasticNetParam': 0.0},
    {'regParam': 0.1,  'elasticNetParam': 0.5},
    {'regParam': 0.5,  'elasticNetParam': 1.0},
]

for params in lr_params:
    with mlflow.start_run(run_name=f"LR_reg{params['regParam']}_en{params['elasticNetParam']}"):

        # Log hyperparameters
        mlflow.log_params({
            'model_type':       'LogisticRegression',
            'regParam':         params['regParam'],
            'elasticNetParam':  params['elasticNetParam'],
            'train_size':       train_df.count(),
        })

        # Build & fit pipeline
        lr = LogisticRegression(
            featuresCol='features', labelCol='label',
            regParam=params['regParam'],
            elasticNetParam=params['elasticNetParam'],
            maxIter=100
        )
        pipeline = build_pipeline(lr)
        model    = pipeline.fit(train_df)

        # Evaluate
        preds  = model.transform(test_df)
        auc    = evaluator.evaluate(preds)
        acc    = preds.filter(F.col('prediction') == F.col('label')).count() / preds.count()

        mlflow.log_metric('auc_roc',  auc)
        mlflow.log_metric('accuracy', acc)

        # Log the fitted Spark pipeline model
        mlflow.spark.log_model(model, artifact_path='spark_model')

        print(f"LR reg={params['regParam']} en={params['elasticNetParam']} → AUC={auc:.4f}  Acc={acc:.4f}")

4b  Gradient-Boosted Trees


In [0]:
gbt_params = [
    {'maxDepth': 3, 'maxIter': 20},
    {'maxDepth': 5, 'maxIter': 50},
    {'maxDepth': 7, 'maxIter': 50},
]

for params in gbt_params:
    with mlflow.start_run(run_name=f"GBT_depth{params['maxDepth']}_iter{params['maxIter']}"):

        mlflow.log_params({
            'model_type': 'GBTClassifier',
            'maxDepth':   params['maxDepth'],
            'maxIter':    params['maxIter'],
            'train_size': train_df.count(),
        })

        gbt = GBTClassifier(
            featuresCol='features', labelCol='label',
            maxDepth=params['maxDepth'],
            maxIter=params['maxIter'],
            seed=42
        )
        pipeline = build_pipeline(gbt)
        model    = pipeline.fit(train_df)

        preds = model.transform(test_df)
        auc   = evaluator.evaluate(preds)
        acc   = preds.filter(F.col('prediction') == F.col('label')).count() / preds.count()

        mlflow.log_metric('auc_roc',  auc)
        mlflow.log_metric('accuracy', acc)

        mlflow.spark.log_model(model, artifact_path='spark_model')

        print(f"GBT depth={params['maxDepth']} iter={params['maxIter']} → AUC={auc:.4f}  Acc={acc:.4f}")

### 5)  Compare Runs & Register the Best Model

In [0]:
# Pull all runs for this experiment
experiment = mlflow.get_experiment_by_name('/Shared/phase5_campaign_model')
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.auc_roc DESC']
)

display(runs_df[['run_id', 'params.model_type', 'params.maxDepth', 'params.maxIter',
                  'params.regParam', 'metrics.auc_roc', 'metrics.accuracy']])

best_run = runs_df.iloc[0]
print(f"\n Best run  : {best_run['run_id']}")
print(f" Model type: {best_run.get('params.model_type')}")
print(f" AUC-ROC   : {best_run['metrics.auc_roc']:.4f}")
print(f" Accuracy  : {best_run['metrics.accuracy']:.4f}")

In [0]:
# Register best model in the MLflow Model Registry
model_uri  = f"runs:/{best_run['run_id']}/spark_model"
model_name = 'campaign_performance_classifier'

registered = mlflow.register_model(model_uri=model_uri, name=model_name)

print(f'Registered model  : {registered.name}')
print(f'Version           : {registered.version}')
print(f'Status            : {registered.status}')

### 6)  Load the Registered Model & Score New Data

In [0]:
# Load directly from the registry – no run ID needed
loaded_model = mlflow.spark.load_model(f'models:/{model_name}/latest')

# Score a fresh batch (here we reuse test_df as a proxy)
scored = loaded_model.transform(test_df)
display(scored.select('brand_source', 'campaign_type', 'roas',
                       'label', 'prediction', 'probability').limit(20))

## Summary

| Step | What was done |
|------|---------------|
| **Data ingestion** | `load_all_brands()` reads, normalises, and unions all 3 brand CSVs automatically |
| **Feature engineering** | `engineer_features()` encapsulates channel flags, ROAS, date parts, and label creation |
| **Spark Pipeline** | `build_pipeline(classifier)` chains StringIndexer → OHE → VectorAssembler → model |
| **MLflow tracking** | Every run logs hyperparameters, AUC-ROC, accuracy, and the full Spark model artefact |
| **Model comparison** | `mlflow.search_runs()` ranks all runs; best is auto-selected |
| **Model Registry** | Best pipeline registered under `campaign_performance_classifier` for re-use |
